# Module 3 Lab 2: Outliers in the Crime Data

Krystal-Rose C. Uduma | CS 82A

Input: `usarrests.csv` (Canvas course file, saved next to this notebook).

Portfolio: https://github.com/krystalroseuduma/cs82a-portfolio

## 1. Summarize the original values

Each row is one state. Murder and Assault are arrest rates per 100,000 residents, and UrbanPop is the percentage living in urban areas.

In [1]:
import pandas as pd

crime = pd.read_csv("usarrests.csv", index_col=0)
crime.index.name = "State"
columns = ["Murder", "Assault", "UrbanPop"]
print(crime[columns].describe().round(2).to_string())
print("\nMissing values by column:")
print(crime[columns].isna().sum().to_string())

       Murder  Assault  UrbanPop
count   50.00    49.00     50.00
mean     7.79   182.18     74.20
std      4.36   130.88     73.41
min      0.80    45.00      6.00
25%      4.08   109.00     53.25
50%      7.25   159.00     66.00
75%     11.25   249.00     77.75
max     17.40   879.00    570.00

Missing values by column:
Murder      0
Assault     1
UrbanPop    0


## 2. Screen for invalid and unusual values

UrbanPop must be between 0 and 100 inclusive, and arrest rates must be nonnegative. I also use the 1.5 IQR fences to find values that are unusual for this file. Crossing a fence prompts a check of the source.

In [2]:
q1 = crime[columns].quantile(0.25)
q3 = crime[columns].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
print(pd.DataFrame({"Lower fence": lower, "Upper fence": upper})
      .round(2).to_string())

unusual = crime[columns].lt(lower) | crime[columns].gt(upper)
invalid = crime[columns].lt(0)
invalid["UrbanPop"] = invalid["UrbanPop"] | crime["UrbanPop"].gt(100)
print("\nFlagged values:")
for col in columns:
    for state, value in crime.loc[unusual[col] | invalid[col], col].items():
        label = "Invalid range" if invalid.loc[state, col] else "Review"
        print(f"{state}: {col} = {value:g} ({label})")
print("\nMissing observations:")
print(crime.loc[crime[columns].isna().any(axis=1)].to_string())

          Lower fence  Upper fence
Murder          -6.69        22.01
Assault       -101.00       459.00
UrbanPop        16.50       114.50

Flagged values:
South Carolina: Assault = 879 (Review)
Iowa: UrbanPop = 570 (Invalid range)
New York: UrbanPop = 6 (Review)

Missing observations:
         Murder  Assault  UrbanPop
State                             
Georgia    17.4      NaN        60


## 3. Verdicts and actions

**Iowa, UrbanPop 570%.** This is impossible because a percentage of residents cannot exceed 100. The published USArrests data list 57%, so I use 57 in the analysis copy.

**New York, UrbanPop 6%.** Six percent falls within the possible range, but it is unusually low in this dataset. I checked the published data and found 86%, so I use that value in the analysis copy.

**South Carolina, Assault 879 per 100,000.** This rate is possible in principle, but it stands far above the other state rates. The published data list 279, so I use 279 in the analysis copy.

**Georgia, missing Assault rate.** A blank gives us no rate to evaluate. The published data provide 211 per 100,000, so I restore that value and keep Georgia in the analysis.

Comparison data: [USArrests public data export](https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/datasets/USArrests.csv).

## 4. Apply the corrections and prepare urban population categories

In [3]:
fixed = crime.copy()
corrections = [
    ("Iowa", "UrbanPop", 57),
    ("New York", "UrbanPop", 86),
    ("South Carolina", "Assault", 279),
    ("Georgia", "Assault", 211),
]
for state, col, corrected in corrections:
    print(f"{state} {col}: {crime.loc[state, col]:g} -> {corrected}")
    fixed.loc[state, col] = corrected

assert len(fixed) == 50 and not fixed[columns].isna().any().any()
assert fixed["UrbanPop"].between(0, 100).all()
fixed["Urban category"] = pd.cut(
    fixed["UrbanPop"], bins=[0, 50, 60, 70, 101], right=False,
    labels=["Small", "Medium", "Large", "Extra-large"],
)
print("\nMean assault rate by urban population category:")
print(fixed.groupby("Urban category", observed=True)["Assault"]
      .agg(states="size", mean_assault="mean").round(2).to_string())

Iowa UrbanPop: 570 -> 57
New York UrbanPop: 6 -> 86
South Carolina Assault: 879 -> 279
Georgia Assault: nan -> 211

Mean assault rate by urban population category:
                states  mean_assault
Urban category                      
Small                8        174.75
Medium               9        127.56
Large               12        153.50
Extra-large         21        197.62


The analysis copy contains all 50 states. I kept the source file intact and repaired only the four entries documented above.